In [ ]:
import pandas as pd

# 1. Cargar el dataset desde la carpeta raw_data
data_path = "raw_data/df_modelo_pf.csv"
df_raw = pd.read_csv(data_path)

# 2. Seleccionar variables clave para nuestro MVP y asignar alias limpios
columnas_mvp = {
    'ingreso_mensual': 'monthly_income',
    'gasto_mensual_total': 'monthly_expenses',
    'tasa_ahorro': 'savings_rate',
    'puntaje_crediticio': 'credit_score',
    'escenario_financiero': 'financial_scenario'
}

# 3. Filtrar y renombrar
df_mvp = df_raw[list(columnas_mvp.keys())].copy()
df_mvp.rename(columns=columnas_mvp, inplace=True)

# 4. Mostrar dimensiones y vista previa
print(f"Dataset listo con {df_mvp.shape[0]} filas y {df_mvp.shape[1]} columnas.\n")
df_mvp.head()

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# 1. Feature Engineering: Crear métricas financieras derivadas
df_mvp['net_margin'] = df_mvp['monthly_income'] - df_mvp['monthly_expenses']
df_mvp['expense_ratio'] = df_mvp['monthly_expenses'] / df_mvp['monthly_income']

# 2. Codificar la variable objetivo de texto a números
label_encoder = LabelEncoder()
df_mvp['target_encoded'] = label_encoder.fit_transform(df_mvp['financial_scenario'])

# Guardar las clases para saber qué número corresponde a qué texto
print("Mapeo de etiquetas:")
for idx, class_name in enumerate(label_encoder.classes_):
    print(f"  {idx} -> {class_name}")

# 3. Definir Matriz de Características (X) y Objetivo (y)
X = df_mvp[['monthly_income', 'monthly_expenses', 'savings_rate', 'credit_score', 'net_margin', 'expense_ratio']]
y = df_mvp['target_encoded']

# 4. Split Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Entrenar el modelo con mayor número de estimadores
model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# 6. Evaluar
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n¡NUEVA PRECISIÓN MEJORADA!: {acc * 100:.2f}%\n")

# 7. Si la precisión es adecuada, empaquetamos el modelo JUNTO con el encoder
models_dir = Path.cwd().parent / "app" / "models"
if not models_dir.exists():
    models_dir = Path.cwd() / "app" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Guardamos un diccionario que contiene el modelo y el encoder de etiquetas
artifact = {
    'model': model,
    'encoder': label_encoder,
    'features': list(X.columns)
}

artifact_path = models_dir / "financial_scenario_model.pkl"
joblib.dump(artifact, artifact_path)
print(f"Artefacto completo guardado exitosamente en: {artifact_path}")

In [ ]:
# 1. Ver qué categorías existen en la variable objetivo
print("--- Distribución del Target (Escenarios) ---")
print(df_mvp['financial_scenario'].value_counts())

print("\n--- Tipos de datos y nulos ---")
print(df_mvp.info())

print("\n--- Primeras filas para revisar formato ---")
df_mvp.head(10)

In [ ]:
# Ver todas las columnas disponibles en el dataset completo
print("Columnas disponibles en df_raw:")
for col in df_raw.columns:
    print(f"- {col}")

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# 1. Copia del dataset completo
df_full = df_raw.copy()

# 2. Identificar la columna target (escenario_financiero)
target_col = 'escenario_financiero' if 'escenario_financiero' in df_full.columns else 'financial_scenario'

# 3. Codificar variables de texto (categóricas) a números
label_encoders = {}
for col in df_full.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df_full[col] = le.fit_transform(df_full[col].astype(str))
    label_encoders[col] = le

# 4. Separar X e y
X = df_full.drop(columns=[target_col])
y = df_full[target_col]

# 5. Split Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 6. Entrenar Random Forest
model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42)
model.fit(X_train, y_train)

# 7. Evaluar
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"¡NUEVA PRECISIÓN CON TODAS LAS COLUMNAS!: {acc * 100:.2f}%\n")

# 8. Guardar si mejoró
if acc > 0.70:
    models_dir = Path.cwd().parent / "app" / "models"
    if not models_dir.exists():
        models_dir = Path.cwd() / "app" / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
    
    artifact = {
        'model': model,
        'encoders': label_encoders,
        'features': list(X.columns)
    }
    joblib.dump(artifact, models_dir / "financial_scenario_model.pkl")
    print("Modelo guardado exitosamente.")

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

df_work = df_raw.copy()

# 1. Crear reglas financieras lógicas para la variable objetivo (Target)
# Definimos el escenario en función de métricas clave:
# - Recesión: El gasto supera al ingreso o la relación deuda/ingreso es crítica (> 0.5)
# - Inflación: La tasa de ahorro es muy baja (< 0.10) o hay nivel de estrés alto
# - Normal: Finanzas estables con buen margen y ahorro aceptable

condiciones = [
    (df_work['gasto_mensual_total'] > df_work['ingreso_mensual']) | (df_work['relacion_deuda_ingreso'] > 0.50),
    (df_work['tasa_ahorro'] < 0.10) | (df_work['nivel_estres_financiero'] == 'Alto'),
]
elecciones = ['Recesión', 'Inflación']

# Si no cumple condiciones de riesgo, se clasifica como 'Normal'
df_work['escenario_financiero'] = np.select(condiciones, elecciones, default='Normal')

print("--- Nueva Distribución Lógica del Target ---")
print(df_work['escenario_financiero'].value_counts())

# 2. Descartar variables sin valor predictivo (IDs, fechas)
columnas_a_descartar = ['id_usuario', 'fecha']
df_work = df_work.drop(columns=[col for col in columnas_a_descartar if col in df_work.columns])

# 3. Separar X e y
target_col = 'escenario_financiero'
y_raw = df_work[target_col]
X_raw = df_work.drop(columns=[target_col])

# 4. Codificar Target y Variables Categóricas
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw.astype(str))

X = X_raw.copy()
feature_encoders = {}
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    feature_encoders[col] = le

# 5. Split Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 6. Entrenar Random Forest
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# 7. Evaluar Precisión
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\n==========================================")
print(f"¡NUEVA PRECISIÓN DEL MODELO!: {acc * 100:.2f}%")
print(f"==========================================\n")

# 8. Exportar Artefacto .pkl Completo
models_dir = Path.cwd().parent / "app" / "models"
if not models_dir.exists():
    models_dir = Path.cwd() / "app" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

artifact = {
    'model': model,
    'target_encoder': target_encoder,
    'feature_encoders': feature_encoders,
    'features': list(X.columns)
}

artifact_path = models_dir / "financial_scenario_model.pkl"
joblib.dump(artifact, artifact_path)
print(f"Artefacto guardado exitosamente en: {artifact_path}")

In [ ]:
import joblib
from pathlib import Path
import pandas as pd

# 1. Definir ruta e intentar cargar el artefacto .pkl
models_dir = Path.cwd().parent / "app" / "models"
if not models_dir.exists():
    models_dir = Path.cwd() / "app" / "models"

artifact_path = models_dir / "financial_scenario_model.pkl"

print(f"Cargando artefacto desde: {artifact_path}")
artifact = joblib.load(artifact_path)

# 2. Desempaquetar componentes
model = artifact['model']
target_encoder = artifact['target_encoder']
feature_encoders = artifact['feature_encoders']
features = artifact['features']

print("✓ Artefacto deserializado con éxito.")
print(f"✓ Modelo cargado: {type(model).__name__}")
print(f"✓ Total de features esperadas: {len(features)}")
print(f"✓ Clases detectadas: {list(target_encoder.classes_)}\n")

# 3. Crear una fila de prueba ficticia (Simulando un perfil de riesgo: Recesión)
# Gasto mayor a ingreso -> Debería predecir "Recesión"
sample_data = {col: [0] for col in features}  # Estructura base

# Asignar valores específicos para las variables clave
sample_data['ingreso_mensual'] = [3000.0]
sample_data['gasto_mensual_total'] = [4500.0] # Gasto > Ingreso
sample_data['relacion_deuda_ingreso'] = [0.60]
sample_data['tasa_ahorro'] = [0.0]
sample_data['nivel_estres_financiero'] = ['Alto']

df_sample = pd.DataFrame(sample_data)

# 4. Transformar variables categóricas de la muestra
for col, le in feature_encoders.items():
    if col in df_sample.columns:
        # Transformar usando el encoder guardado (si no existe el valor, fallback a 0)
        df_sample[col] = df_sample[col].astype(str).map(
            lambda s: le.transform([s])[0] if s in le.classes_ else 0
        )

# 5. Predecir
pred_numeric = model.predict(df_sample[features])
pred_label = target_encoder.inverse_transform(pred_numeric)

print("--- RESULTADO DEL TEST ---")
print(f"Predicción numérica: {pred_numeric[0]}")
print(f"Escenario predicho: {pred_label[0]}")

## Plan de Acción en el Notebook para Cumplir al 100%
Para dejar la solución alineada con la especificación técnica, adaptaremos el notebook agregando dos componentes:

1- Modelo NLP / Clasificador de Transacciones:

* Entrenar un modelo liviano (**TF-IDF** + **MultinomialNB** o **LogisticRegression**) para mapear textos como "Supermercado", "Uber", "Farmacia" a categorías (Alimentación, Transporte, Salud, Ocio, Servicios).

2- Re-mapeo del Modelo de Perfil Financiero:

* Mapear las clases del objetivo a: Saludable, En observación, En riesgo.

* Agregar un serializador secundario o un artefacto único que guarde ambos modelos (**classifier_nlp.pkl** y **perfil_model.pkl**).

In [ ]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# 1. ENTRENAR CLASIFICADOR DE TRANSACCIONES (NLP)
# ==========================================
# Dataset de entrenamiento para categorización de gastos
datos_transacciones = [
    ("Supermercado Coto", "Alimentación"), ("Compra en almacén", "Alimentación"), 
    ("Verdulería central", "Alimentación"), ("Restaurante cena", "Alimentación"),
    ("Combustible YPF", "Transporte"), ("Ticket de subte", "Transporte"), 
    ("Carga de nafta", "Transporte"), ("Uber viaje", "Transporte"),
    ("Farmacia consulta", "Salud"), ("Remedios osde", "Salud"),
    ("Pago de alquiler", "Vivienda"), ("Expensas departamento", "Vivienda"),
    ("Cuota universidad", "Educación"), ("Curso udemy", "Educación"),
    ("Streaming cine", "Ocio"), ("Netflix suscripcion", "Ocio"), ("Spotify mensual", "Ocio"),
    ("Pago luz edesur", "Servicios"), ("Factura de agua", "Servicios"), ("Internet fibra", "Servicios")
]

df_nlp = pd.DataFrame(datos_transacciones, columns=['descripcion', 'categoria'])

# Creación del Pipeline TF-IDF + Classifier
nlp_pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    MultinomialNB()
)
nlp_pipeline.fit(df_nlp['descripcion'], df_nlp['categoria'])

print("✓ Clasificador NLP de gastos entrenado correctamente.")

# ==========================================
# 2. MODELO DE PERFIL FINANCIERO (CON TAXONOMÍA SOLICITADA)
# ==========================================
df_profile = df_raw.copy()

# Mapeo a las categorías requeridas: Saludable, En observación, En riesgo
condiciones = [
    (df_profile['gasto_mensual_total'] > df_profile['ingreso_mensual']) | (df_profile['relacion_deuda_ingreso'] > 0.50),
    (df_profile['tasa_ahorro'] < 0.10) | (df_profile['nivel_estres_financiero'] == 'Alto')
]
elecciones = ['En riesgo', 'En observación']
df_profile['perfil_financiero'] = np.select(condiciones, elecciones, default='Saludable')

# Preparar variables para entrenamiento del perfil
X_profile = df_profile[['ingreso_mensual', 'gasto_mensual_total', 'relacion_deuda_ingreso', 'tasa_ahorro']]
y_profile = df_profile['perfil_financiero']

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y_profile)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_profile, y_encoded)

print("✓ Modelo de Perfil Financiero ajustado a las nuevas etiquetas.")

# ==========================================
# 3. GUARDAR ARTEFACTOS SERIALIZADOS
# ==========================================
models_dir = Path.cwd().parent / "app" / "models"
if not models_dir.exists():
    models_dir = Path.cwd() / "app" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Exportar modelo NLP
joblib.dump(nlp_pipeline, models_dir / "transaction_categorizer.pkl")

# Exportar modelo de Perfil
artifact_perfil = {
    'model': rf_model,
    'target_encoder': target_encoder,
    'features': list(X_profile.columns)
}
joblib.dump(artifact_perfil, models_dir / "financial_profile_model.pkl")

print(f"✓ Artefactos serializados exitosamente en: {models_dir}")